# grounded-reasoning — quickstart

A **relation-algebra verifier** for LLMs and agents: check a multi-hop relational claim *before* asserting it — **zero model tokens**, **precision guaranteed** (a claim is accepted iff a grounded proof path exists).

Everything in this notebook runs **offline** — no API key, no LLM call. That is the point: the verification layer is local matrix algebra.

Repo: https://github.com/ALEXaquarius/grounded-reasoning · Full paper: [PAPER.md](https://github.com/ALEXaquarius/grounded-reasoning/blob/main/PAPER.md) · Agent integration: [docs/integration.md](https://github.com/ALEXaquarius/grounded-reasoning/blob/main/docs/integration.md)

In [ ]:
%pip install -q grounded-reasoning

## 1. Verify a multi-hop claim

Load one-hop facts, then ask about a *composed* relation. The verdict comes with a **proof path** — or a rejection when no path exists (exactly where LLMs fabricate).

In [ ]:
from grounded_reasoning import GroundedReasoner

gr = GroundedReasoner()
gr.add_facts([
    ("alice", "parent", "bob"),
    ("bob",   "parent", "carol"),
    ("carol", "parent", "dave"),
])

print(gr.verify("alice", "dave", via="parent"))   # 3-hop ancestor: grounded, with proof
print(gr.verify("alice", "zed",  via="parent"))   # no path: blocked (this is a hallucination in an LLM's mouth)

## 2. Filter a batch of LLM claims

The typical integration: an LLM emits relational claims, and only the ones with an evidence path survive.

In [ ]:
llm_claims = [
    ("alice", "carol", "parent"),   # true 2-hop composition
    ("alice", "zed",   "parent"),   # fabricated
    ("bob",   "dave",  "parent"),   # true 2-hop composition
]

for claim, verdict in gr.filter_claims(llm_claims):
    status = "KEPT   " if verdict.grounded else "BLOCKED"
    print(f"{status} {claim}  proof={verdict.proof}")

## 3. Detect contradictions — for free

If a relation should be a hierarchy (is-a, part-of, causes) but the asserted facts contain a **cycle**, that is a certificate of contradiction. Detection is O(V+E), zero tokens.

In [ ]:
gr2 = GroundedReasoner()
gr2.add_facts([
    ("cat",    "is_a", "mammal"),
    ("mammal", "is_a", "animal"),
    ("animal", "is_a", "cat"),     # an LLM asserted a contradictory loop
])
print("contradictory cycle:", gr2.contradictions("is_a"))

## 4. Any language, zero configuration

Entities and relations are opaque Unicode strings — the algebra does not care what language they are in.

In [ ]:
gr3 = GroundedReasoner()
gr3.add_facts([("Anh", "cha", "B\u1ea3o"), ("B\u1ea3o", "cha", "C\u01b0\u1eddng")])   # Vietnamese
print(gr3.verify("Anh", "C\u01b0\u1eddng", via="cha").proof)

gr4 = GroundedReasoner()
gr4.add_facts([("\u7236", "\u662f", "\u7956\u7236")])                                  # Chinese
print(gr4.verify("\u7236", "\u7956\u7236", via="\u662f").grounded)

## 5. Noisy graphs — the conformal mode

The hard guard assumes the graph is correct. When facts are extracted by an LLM from raw text, edges go missing. `ConformalReasoner` trades the hard precision guarantee for a **distribution-free coverage guarantee ≥ 1−α** that survives noise (Theorem K).

In [ ]:
from grounded_reasoning import ConformalReasoner
from src.reasoning.abstract_inference import FuzzyInferenceEngine
import random

rng = random.Random(0)
true_edges = [(f"c{i}", f"c{i+1}") for i in range(60)]

# a noisy copy of the graph: 20% of true edges were lost at extraction time
eng = FuzzyInferenceEngine(walk_len=8, alpha=0.7)
for a, b in true_edges:
    if rng.random() > 0.2:
        eng.add_relation(a, b)

true_pairs = [(f"c{i}", f"c{i+2}") for i in range(0, 50)]   # true 2-hop facts
rng.shuffle(true_pairs)   # calibration/test must be exchangeable — always shuffle before splitting
cr = ConformalReasoner(eng, alpha=0.1)
tau = cr.calibrate(true_pairs[:25])                          # calibrate on half
kept = sum(cr.accept(a, b) for a, b in true_pairs[25:])     # test on the other half
print(f"threshold tau={tau:.4g};  coverage on held-out true pairs: {kept}/25 (guarantee: >= 0.9 on average)")

## 6. The theorems behind the guarantees

Every guarantee above is a theorem with a numerical verification shipped in the package — run them all right now:

In [ ]:
from src.theory.theorems import ALL_THEOREMS

for fn in ALL_THEOREMS:
    r = fn()
    ok = "CONFIRMED" in r["conclusion"]
    print(f"{'PASS' if ok else 'FAIL'}  {r['theorem']}")

## 7. Using it from an agent

Three integration paths — a Python library (above), a stateless **function-calling tool** (Anthropic/OpenAI schemas below), or an **MCP server** (`python -m src.agent.mcp_server`). Details: [docs/integration.md](https://github.com/ALEXaquarius/grounded-reasoning/blob/main/docs/integration.md).

In [ ]:
from grounded_reasoning import TOOL_SPEC, run_tool

print("tool name:", TOOL_SPEC["name"])

# exactly what an agent would send when the model calls the tool:
result = run_tool({
    "facts": [["aspirin", "treats", "headache"], ["headache", "symptom_of", "flu"]],
    "subject": "aspirin", "relation": "treats", "object": "flu",
})
print(result)   # grounded=False: 'aspirin treats flu' is NOT derivable from these facts

## Going further — real-LLM experiments

The repo ships the experiments behind every number in the README (guard 33%→100% on DeepSeek, SGDC, CLUTRR, end-to-end conformal over an LLM-extracted graph). They need an API key (read from an environment variable, never hardcoded):

```bash
export DEEPSEEK_API_KEY=sk-...
python -m src.experiments.guard_llm_eval
python -m src.experiments.clutrr_eval
```

And don't take any of this on faith — the full offline test suite locks every claim: `pytest tests/` (116 tests, no key needed).